# 14 — Full-event evaluation

**Two models** — the naive baseline and the disjoint-heads model (v5) —
evaluated **two ways**:

- **teacher-forced** — when predicting field *k*, the true values of fields
  *0…k-1* of the same event are given.
- **autoregressive** — predict field 0, feed *that prediction* back in, predict
  field 1, … through `result`. Only the 3-event history and the always-on match
  state (`minute`, `second`, goal/card counts) are true.

The naive baseline predicts a fixed constant per field regardless of context,
so its two modes come out identical — both are still computed and shown, so the
table is a clean 2 models × 2 modes grid.

Metrics: accuracy for `is_home`/`event`/`result`, MAE otherwise. Every event in
the split, boundary events included, argmax (greedy) predictions. (Cross-entropy
is also computed, but only as a sanity check that this eval path matches
training — not a reported result.)

In [1]:
# ── the one knob ──────────────────────────────────────────────────────────
# "val" while iterating. Set to "test" ONCE, for the final held-out number,
# then commit the run.
SPLIT = "val"
# ─────────────────────────────────────────────────────────────────────────────

import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

train = pd.read_csv("data/train_set.csv")
split = pd.read_csv(f"data/{SPLIT}_set.csv")
print(f"evaluating on: {SPLIT}  ({len(split):,} rows, {split['game_id'].nunique()} games)")

event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]
categorical = ["is_home", "event", "result"]
continuous_fields = ["start_x", "start_y", "time_elapsed", "duration"]
match_state_fields = [
    "period_id", "minute", "second",
    "home_goal_count", "away_goal_count",
    "home_red_count", "away_red_count",
    "home_yellow_count", "away_yellow_count",
]
embed_fields = ["is_home", "event", "result"]


def with_lags(fields, n_lags=3):
    cols = list(fields)
    for lag in range(1, n_lags + 1):
        cols += [f"{f}-{lag}" for f in fields]
    return cols


continuous_cols = with_lags(continuous_fields) + match_state_fields
embed_cols = {f: [f] + [f"{f}-{lag}" for lag in range(1, 4)] for f in embed_fields}

with open("data/event_masks.json") as f:
    raw_masks = json.load(f)
with open("data/category_lookup.json") as f:
    lookup = {c: {int(k): v for k, v in m.items()} for c, m in json.load(f).items()}

evaluating on: val  (523,817 rows, 38 games)


## Shared preprocessing (matches v5 training exactly)

In [2]:
def remap_sentinel(values):
    values = np.asarray(values, dtype=float)
    values = np.where(values == 106, -2.0, values)
    return (values + 2).astype(np.int64)


def normalize_field(values, mean, std):
    values = np.asarray(values, dtype=float)
    is_sentinel = (values == -1) | (values == -2)
    return np.where(is_sentinel, values, (values - mean) / std + 10)


# stats from train only, sentinels excluded — same as the v5 notebook
continuous_stats = {}
for field in continuous_fields:
    cols = [field] + [f"{field}-{lag}" for lag in range(1, 4)]
    v = train[cols].values.flatten().astype(float)
    v = v[(v != -1) & (v != -2)]
    continuous_stats[field] = (v.mean(), v.std())
for field in ["minute", "second"]:
    v = train[field].values.astype(float)
    v = v[(v != -1) & (v != -2)]
    continuous_stats[field] = (v.mean(), v.std())

masks = {c: remap_sentinel(v).tolist() for c, v in raw_masks.items()}
shifted_of = {f: torch.tensor(masks[f]) for f in event_cols}          # local idx -> shifted code
local_of = {}
for f in event_cols:
    pl = torch.full((108,), -1, dtype=torch.long)
    pl[torch.tensor(masks[f])] = torch.arange(len(masks[f]))
    local_of[f] = pl                                                  # shifted code -> local idx

In [3]:
def build_cont(df):
    X = df[continuous_cols].values.astype(float).copy()
    for field in continuous_fields:
        m, s = continuous_stats[field]
        for col in [field] + [f"{field}-{lag}" for lag in range(1, 4)]:
            j = continuous_cols.index(col)
            X[:, j] = normalize_field(X[:, j], m, s)
    for field in ["minute", "second"]:
        m, s = continuous_stats[field]
        j = continuous_cols.index(field)
        X[:, j] = normalize_field(X[:, j], m, s)
    return torch.tensor(X).float()


def build_embed(df):
    return {f: torch.tensor(remap_sentinel(df[c].values)).long() for f, c in embed_cols.items()}


def field_targets(df, field):
    y = df.loc[df["step"] == event_cols.index(field), "target"].to_numpy()
    return np.where(y == 106, -2, y)                        # boundary sentinel back to -2


def true_local(field, df):
    return local_of[field][torch.tensor(remap_sentinel(field_targets(df, field)))]

## The disjoint-heads model (v5)

In [4]:
ckpt = torch.load("data/mlp_embeddings_disjoint_checkpoint.pt", weights_only=False)
C = {"event": ckpt["C_event"], "is_home": ckpt["C_is_home"], "result": ckpt["C_result"]}
W1, b1, W2, b2, W3, b3 = ckpt["W1"], ckpt["b1"], ckpt["W2"], ckpt["b2"], ckpt["W3"], ckpt["b3"]
output_heads = ckpt["output_heads"]


def trunk(X_cont, X_embed):
    embs = [C[k][X_embed[k]].reshape(X_embed[k].shape[0], -1) for k in embed_fields]
    h = torch.cat([X_cont, *embs], dim=1)
    h = torch.relu(h @ W1 + b1)
    h = torch.relu(h @ W2 + b2)
    h = torch.relu(h @ W3 + b3)
    return h


@torch.no_grad()
def head_logits(X_cont, X_embed, field):
    return trunk(X_cont, X_embed) @ output_heads[field][0] + output_heads[field][1]


def to_raw(field, local_idx):
    return (shifted_of[field][local_idx] - 2).numpy()      # -> -2 boundary, 0.. real

### Teacher-forced

Each `step == k` row already has fields `0…k-1` filled with their true values
and `k…6` masked — exactly the teacher-forced input for predicting field `k`.

In [5]:
def v5_teacher_forced():
    preds, ces = {}, {}
    for field in event_cols:
        rows = split[split["step"] == event_cols.index(field)]
        logits = head_logits(build_cont(rows), build_embed(rows), field)
        preds[field] = to_raw(field, logits.argmax(1))
        ces[field] = F.cross_entropy(logits, true_local(field, rows)).item()
    return preds, ces

### Autoregressive

Start from every event's `step == 0` row (all current fields masked), then walk
`is_home → … → result`, writing each argmax prediction back into the input
before predicting the next field. Cross-entropy at each step is still measured
against the *true* target — "how surprised is the model by the truth, given the
context it generated itself".

In [6]:
def v5_autoregressive():
    start = split[split["step"] == 0].copy()
    X_cont, X_embed = build_cont(start), build_embed(start)

    preds, ces = {}, {}
    for field in event_cols:
        logits = head_logits(X_cont, X_embed, field)
        pred_raw = to_raw(field, logits.argmax(1))
        preds[field] = pred_raw
        ces[field] = F.cross_entropy(logits, true_local(field, split)).item()

        shifted = torch.tensor(remap_sentinel(pred_raw))
        if field in embed_fields:
            X_embed[field] = X_embed[field].clone()
            X_embed[field][:, 0] = shifted
        else:
            m, s = continuous_stats[field]
            X_cont = X_cont.clone()
            X_cont[:, continuous_fields.index(field)] = torch.tensor(normalize_field(pred_raw, m, s)).float()
    return preds, ces

## Naive baseline (mode / median / training marginal)

In [7]:
baseline_pred, baseline_ce = {}, {}
for field in event_cols:
    tr = field_targets(train, field)
    if field in categorical:
        vals, counts = np.unique(tr, return_counts=True)
        baseline_pred[field] = int(vals[counts.argmax()])
    else:
        baseline_pred[field] = float(np.median(tr))

    classes = list(masks[field])                      # already-shifted legal codes
    pos = {c: i for i, c in enumerate(classes)}
    counts = np.bincount([pos[v] for v in remap_sentinel(tr)], minlength=len(classes)).astype(float)
    p = (counts + 1) / (counts + 1).sum()
    y = remap_sentinel(field_targets(split, field))
    baseline_ce[field] = float(np.mean([-np.log(p[pos[v]]) for v in y]))

baseline_pred

{'is_home': 1,
 'event': 1,
 'start_x': 54.0,
 'start_y': 34.0,
 'time_elapsed': 1.0,
 'duration': 1.0,
 'result': 0}

## The table

2 models × 2 modes. Within each mode, the better model per field is bolded
(higher accuracy / lower MAE).

In [8]:
def score(pred_raw, field):
    truth = field_targets(split, field)
    p = np.asarray(pred_raw)
    if field in categorical:
        return float(np.mean(p == truth))
    return float(np.mean(np.abs(p - truth)))


n = {f: len(field_targets(split, f)) for f in event_cols}
base_pred = {f: np.full(n[f], baseline_pred[f]) for f in event_cols}   # constant, mode-independent

v5_tf_pred, v5_tf_ce = v5_teacher_forced()
v5_ar_pred, v5_ar_ce = v5_autoregressive()

preds = {
    ("autoregressive", "naive"): base_pred,
    ("autoregressive", "v5"): v5_ar_pred,
    ("teacher-forced", "naive"): base_pred,
    ("teacher-forced", "v5"): v5_tf_pred,
}

records = {}
for field in event_cols:
    metric = "acc" if field in categorical else "mae"
    records[(field, metric)] = {col: score(preds[col][field], field) for col in preds}

table = pd.DataFrame(records).T
table.index.names = ["field", "metric"]
table.columns = pd.MultiIndex.from_tuples(table.columns, names=["mode", "model"])
table = table[["autoregressive", "teacher-forced"]]


def bold_better(row):
    metric = row.name[1]
    out = pd.Series("", index=row.index)
    for mode in ["autoregressive", "teacher-forced"]:
        nv, v5 = row[(mode, "naive")], row[(mode, "v5")]
        v5_wins = (v5 > nv) if metric == "acc" else (v5 < nv)
        out[(mode, "v5" if v5_wins else "naive")] = "font-weight: bold"
    return out


table.style.apply(bold_better, axis=1).format(precision=3)

### Which model wins, by mode

In [9]:
for mode in ["autoregressive", "teacher-forced"]:
    wins = []
    for (field, metric), r in table.iterrows():
        v5_wins = (r[(mode, "v5")] > r[(mode, "naive")]) if metric == "acc" \
            else (r[(mode, "v5")] < r[(mode, "naive")])
        wins.append(field if v5_wins else f"{field}*")
    n_v5 = sum(1 for w in wins if not w.endswith("*"))
    print(f"{mode:16s}  v5 better on {n_v5}/7   (* = naive better: "
          f"{', '.join(w[:-1] for w in wins if w.endswith('*')) or 'none'})")

autoregressive    v5 better on 6/7   (* = naive better: result)
teacher-forced    v5 better on 7/7   (* = naive better: none)


### Sanity check (not a reported result)

The teacher-forced per-field cross-entropies should sum to the training-time
val loss (~9.57 on val) — confirms this eval path matches how the model was
trained.

In [10]:
print(f"v5 teacher-forced summed CE ({SPLIT}): {sum(v5_tf_ce.values()):.3f}")
print(f"v5 autoregressive summed CE ({SPLIT}): {sum(v5_ar_ce.values()):.3f}")

v5 teacher-forced summed CE (val): 9.573
v5 autoregressive summed CE (val): 11.243


### Read-out

- The two **naive** columns are identical by construction — mode/median never
  looks at earlier fields, so autoregressive vs teacher-forced makes no
  difference to it.
- v5 beats naive everywhere under teacher-forcing. Under autoregression the one
  field it loses is `result` — last in the chain, conditioning on six of its
  own (sometimes wrong) guesses, against an "always success" prior that's
  already right ~83% of the time.
- Headline for the write-up: the **autoregressive** block. Teacher-forced is the
  per-conditional upper bound for the results section.